# Tratamento dos dados

Cada passo de tratamento fica dentro da função `preparar(df)`, aplicada **da mesma forma** ao treino e ao teste.
No fim, os dados tratados são gravados em `../dados/` para os notebooks dos modelos os usarem diretamente.

Regras:
- Decisões que dependem dos dados (ex: que colunas são constantes, que categorias existem) são tomadas **só com o treino** e depois aplicadas ao teste.
- **Nunca reordenar nem filtrar linhas do teste**: o `RowId` da submissão é a ordem das linhas.

Passos:
1. Remover colunas com um só valor
2. Remover identificadores (`PID`, `HHID`)
3. Unificar "NR" com "Sem resposta"
4. Converter os números guardados como texto
5. Mapear as escalas ordinais para números
6. Restante texto → `category`
7. Verificar e gravar

In [1]:
import numpy as np
import pandas as pd

train = pd.read_csv("notebooks/dataset_train.csv")
test = pd.read_csv("notebooks/dataset_test_X.csv")

print("Treino:", train.shape, "| Teste:", test.shape)

Treino: (10962, 237) | Teste: (3655, 236)


## Passo 1 — Remover colunas com um só valor

Uma coluna em que todas as linhas têm o mesmo valor não dá nenhuma informação ao modelo.

- `nunique(dropna=False)` conta os valores diferentes **incluindo `NaN`** (uma coluna meio vazia e meio "Sim" tem 2 valores, não 1).
- A lista é calculada no **treino**; depois confirmamos que no teste essas colunas também são constantes **e com o mesmo valor**.

In [2]:
constantes = [c for c in train.columns if train[c].nunique(dropna=False) == 1]
print("Colunas constantes no treino:", constantes)

for c in constantes:
    print(f"  {c}: treino = {train[c].unique()} | teste = {test[c].unique()}")

Colunas constantes no treino: ['PROXY', 'COUNTRY']
  PROXY: treino = [1.] | teste = [1.]
  COUNTRY: treino = ['PT'] | teste = ['PT']


## Passo 2 — Remover identificadores

`PID` (um número diferente por pessoa) e `HHID` (derivado do `PID`: `HHID // 100 == PID`) não descrevem a pessoa.
Se ficassem, o modelo podia encontrar padrões por acaso nos números (ex: "PIDs entre 3000 e 3050 são da faixa 7"),
que não se repetem no teste — não há nenhum `PID` em comum entre treino e teste.

Teste feito com o HistGradientBoosting (mesmos 5 folds): com `PID`/`HHID` Macro F1 = 0,5185 ± 0,014; sem eles 0,5178 ± 0,018
→ diferença dentro do ruído, confirmando que não trazem informação.

In [3]:
IDENTIFICADORES = ["PID", "HHID"]
print("PIDs em comum treino/teste:", len(set(train["PID"]) & set(test["PID"])))

PIDs em comum treino/teste: 0


## Passo 3 — Unificar "NR" com "Sem resposta"

Os dois códigos significam o mesmo (a pessoa não respondeu = **falta verdadeira**). "NR" só aparece em `IN19`, `PA5` e `SK5`.

In [4]:
print({c: (train[c] == "NR").sum() for c in train.columns if (train[c] == "NR").any()})

{'IN19': np.int64(725), 'PA5': np.int64(340), 'SK5': np.int64(62)}


## Passo 4 — Converter os números guardados como texto

Há 31 colunas que são números (altura, peso, nº de pessoas, anos, minutos...) mas vêm como texto porque misturam códigos.
Se ficarem como texto, o modelo trata cada número como uma categoria separada (ex: `BM1` teria 66 categorias)
e **não percebe que 160 cm está perto de 161 cm** — faz cortes complicados e sobreajusta.

O que cada código de texto passa a ser:

| Texto | Tratamento | Porquê |
|---|---|---|
| ">=7", ">= 25", "53 ou mais", ... | o número do limite (7, 25, 53...) | é o valor mínimo; pouca gente está acima |
| "Não aplicável" | `NaN` **+ coluna indicadora** `<coluna>_nao_aplicavel` (0/1) | a pergunta não foi feita — é **informação** (ex: revela sexo/idade), não pode perder-se |
| "Sem resposta" | `NaN` | falta verdadeira |
| "Gravidez sem termo (n.a.)" (`IN24`) | `NaN` + indicadora `IN24_gravidez_sem_termo` | é uma resposta, não falta |

Casos especiais:
- `PE7`, `PE9` estão em **HHMM** (`"0130"` = 1h30) → convertidos para **minutos** (`horas*60 + minutos`).
- `PA1` mistura datas (`2018`, `201910`) e texto ("Nunca", "Há mais de 1 ano") → nova coluna numérica `PA1_meses`
  (meses entre a data indicada e a data da entrevista, `REFYEAR`/`REFMONTH`; se só tem o ano, assume junho).
  A coluna `PA1` fica categórica com "Nunca", "Há mais de 1 ano", "Sem resposta" ou "Data indicada".

As indicadoras só são criadas onde "Não aplicável" aparece em pelo menos 1% do treino (escolhido no treino).

In [5]:
NUMERICAS = ["IN2", "IN3", "IN4", "HHNBPERS_COD", "HHNBPERS_0_13_COD", "HO12_COD", "HO34_COD", "AM3", "AM5",
             "IN24", "IN25", "IN26", "BM1", "BM2", "PE2", "PE4", "PE6", "PE7", "PE8", "PE9", "IN29",
             "DH2", "DH4", "SK2", "IN45", "IN47", "IN55", "IN56", "IN57", "IN59"]   # + PA1 (tratada à parte)

LIMITES = {
    "HHNBPERS_COD": {">=7": "7"},
    "HHNBPERS_0_13_COD": {">=3": "3"},
    "HO12_COD": {">= 25": "25"},
    "HO34_COD": {">= 60": "60"},
    "IN25": {"53 ou mais": "53"},
    "IN26": {"33 ou mais": "33"},
}

HHMM = ["PE7", "PE9"]

# Textos que podem aparecer nestas colunas depois de substituir os limites (tudo o resto seria um erro)
TEXTOS_ESPERADOS = {"Não aplicável", "Sem resposta", "NR", "Gravidez sem termo (n.a.)"}

# Colunas com "Não aplicável" frequente (>= 1% no treino) -> recebem indicadora
COM_NAO_APLICAVEL = [c for c in NUMERICAS if train[c].eq("Não aplicável").mean() >= 0.01]
print("Indicadoras de 'Não aplicável' a criar:", len(COM_NAO_APLICAVEL))
print(COM_NAO_APLICAVEL)

Indicadoras de 'Não aplicável' a criar: 18
['IN2', 'IN3', 'IN4', 'AM3', 'AM5', 'IN24', 'IN25', 'IN26', 'PE7', 'DH2', 'DH4', 'SK2', 'IN45', 'IN47', 'IN55', 'IN56', 'IN57', 'IN59']


In [6]:
def converter_numeros(df):
    for c in NUMERICAS:
        s = df[c].replace(LIMITES.get(c, {}))

        # Segurança: não pode haver texto inesperado (to_numeric transformá-lo-ia em NaN sem avisar)
        num = pd.to_numeric(s, errors="coerce")
        inesperados = set(s[num.isna() & s.notna()]) - TEXTOS_ESPERADOS
        assert not inesperados, f"{c}: textos inesperados {inesperados}"

        # Indicadoras (criadas ANTES de converter, senão perdia-se a informação)
        if c in COM_NAO_APLICAVEL:
            df[f"{c}_nao_aplicavel"] = s.eq("Não aplicável").astype("int8")
        if c == "IN24":
            df["IN24_gravidez_sem_termo"] = s.eq("Gravidez sem termo (n.a.)").astype("int8")

        if c in HHMM:                      # "0130" -> 90 minutos
            num = (num // 100) * 60 + num % 100
        df[c] = num
    return df


def converter_pa1(df):
    s = df["PA1"]
    num = pd.to_numeric(s, errors="coerce")
    ano = np.where(num > 9999, num // 100, num)      # 201910 -> 2019 ; 2018 -> 2018
    mes = np.where(num > 9999, num % 100, 6)         # 201910 -> 10   ; 2018 -> junho
    entrevista = df["REFYEAR"] * 12 + df["REFMONTH"]
    df["PA1_meses"] = entrevista - (ano * 12 + mes)  # NaN quando PA1 é texto
    df["PA1"] = s.where(num.isna(), "Data indicada")
    return df

## Passo 5 — Mapear as escalas ordinais para números

Muitas perguntas têm respostas **com ordem** (ex: Nenhuma → Alguma → Muita dificuldade → Não consegue).
Como `category`, o modelo não sabe essa ordem e tem de a aprender sozinho com os dados — gasta dados e cortes.
Como número (0, 1, 2, 3), um corte do tipo "≤ 1" separa logo "sem ou pouca dificuldade" de "muita".

Teste com o HistGradientBoosting (mesmos 5 folds):

| Hiperparâmetros | Sem mapear | Ordinais mapeadas |
|---|---|---|
| por defeito | 0,5711 ± 0,012 | **0,5827** ± 0,008 (melhor nos 5 folds) |
| procura 2 | 0,5828 ± 0,002 | **0,5891** ± 0,013 |

Regras (iguais às das colunas numéricas):
- "Sem resposta" → `NaN` (falta verdadeira);
- "Não aplicável" → `NaN` **+ indicadora** `<coluna>_nao_aplicavel` (onde aparece em ≥ 1% do treino);
- respostas que não cabem na escala mas são informação → **indicadora própria**:
  "Não aplicável (nunca tentou...)" (`HA1*`) → `_nunca_tentou`; "Não consegue avaliar" (`SS2`) → `_nao_consegue_avaliar`;
  ex-consumidores de álcool (`AL1`, `AL6`) → valor 0 (não bebe) + `_deixou_alcool`;
- se aparecer um valor que não está no dicionário, o código pára (`assert`) — nada é perdido sem aviso.


In [7]:
DIFICULDADE = {"Nenhuma dificuldade": 0, "Alguma dificuldade": 1, "Muita dificuldade": 2, "Não consegue de todo": 3}
SAUDE = {"Muito mau": 0, "Mau": 1, "Razoável": 2, "Bom": 3, "Muito bom": 4}
MENTAL = {"Nunca": 0, "Vários dias": 1, "Mais do que metade dos dias": 2, "Quase todos os dias": 3}
ALIMENTACAO = {"Nunca": 0, "Menos de uma vez por semana": 1, "1 a 3 vezes por semana": 2,
               "4 a 6 vezes por semana": 3, "Uma vez ou mais por dia": 4}
CONCORDANCIA = {"Totalmente em desacordo": 0, "Em desacordo": 1, "Mais ou menos em desacordo": 2,
                "Nem de acordo nem em desacordo": 3, "Mais ou menos de acordo": 4, "De acordo": 5,
                "Totalmente de acordo": 6}
DIAS_ALCOOL = {"Em nenhum dos dias": 0, "1 dia": 1, "2 dias": 2, "3 dias": 3, "4 dias": 4}
BEBIDAS = {"0 bebidas por dia": 0, "1 bebida por dia": 1, "2 bebidas por dia": 2, "3 bebidas por dia": 3,
           "4 a 5 bebidas por dia": 4, "6 a 9 bebidas por dia": 5, "10 a 15 bebidas por dia": 6,
           "16 ou mais bebidas por dia": 7}
FREQ_ALCOOL = {"Nunca consumiu, ou só ocasionalmente para provar": 0,
               "Não consumiu nos últimos 12 meses, por ter deixado de consumir álcool": 0,
               "Menos de uma vez por mês": 1, "Uma vez por mês": 2, "2 a 3 dias por mês": 3,
               "1 a 2 dias por semana": 4, "3 a 4 dias por semana": 5, "5 a 6 dias por semana": 6,
               "Todos os dias ou quase todos os dias": 7}
DURACAO = {"10 a 29 minutos": 0, "30 a 59 minutos": 1, "1 hora a menos de 2 horas": 2,
           "2 horas a menos de 3 horas": 3, "3 horas ou mais": 4}
PA_3_5 = {"Nos últimos 12 meses": 0, "Entre 1 ano e menos de 3 anos": 1, "Entre 3 anos e menos de 5 anos": 2,
          "5 anos ou mais": 3, "Nunca": 4}
PA_2_3 = {"Nos últimos 12 meses": 0, "Entre 1 ano e menos de 2 anos": 1, "Entre 2 anos e menos de 3 anos": 2,
          "3 anos ou mais": 3, "Nunca": 4}
PA_5_10 = {"Nos últimos 12 meses": 0, "Entre 1 ano e menos de 5 anos": 1, "Entre 5 anos e menos de 10 anos": 2,
           "10 anos ou mais": 3, "Nunca": 4}
PA_2_3_5 = {"Nos últimos 12 meses": 0, "Entre 1 ano e menos de 2 anos": 1, "Entre 2 anos e menos de 3 anos": 2,
            "Entre 3 anos e menos de 5 anos": 3, "5 anos ou mais": 4, "Nunca": 5}

ESCALAS = {
    **{c: DIFICULDADE for c in ["PL2", "PL4", "PL5", "IN8", "PL6", "IN9", "PL7", "PL8", "PL9_tot",
                                "PC1A", "PC1B", "PC1C", "PC1D", "PC1E", "IN15",
                                "HA1A", "HA1B", "HA1C", "HA1D", "HA1E", "HA1F", "HA1G"]},
    "HS1": SAUDE, "CD2": SAUDE,
    "HS3": {"Nada limitado": 0, "Limitado mas não severamente": 1, "Severamente limitado": 2},
    **{f"MH1{l}": MENTAL for l in "ABCDEFGH"},
    "PN1": {"Nenhuma": 0, "Muito ligeira": 1, "Ligeira": 2, "Moderada": 3, "Intensa": 4, "Muito intensa": 5},
    "PN2": {"Nada": 0, "Um pouco": 1, "De forma moderada": 2, "Bastante": 3, "De forma extrema": 4},
    **{c: ALIMENTACAO for c in ["DH1", "DH3", "DH5", "DH6", "IN68", "IN69", "IN70", "IN71"]},
    **{c: CONCORDANCIA for c in ["IN50", "IN51", "IN52", "IN53", "IN54"]},
    "AL1": FREQ_ALCOOL, "AL6": FREQ_ALCOOL,
    "AL2": DIAS_ALCOOL, "AL4": DIAS_ALCOOL, "AL3": BEBIDAS, "AL5": BEBIDAS,
    "PE3": DURACAO, "PE5": DURACAO,
    "SK5": {"Nunca ou quase nunca": 0, "Menos de uma vez por semana": 1,
            "Pelo menos uma vez por semana (mas não diariamente)": 2,
            "Diariamente, menos de 1 hora por dia": 3, "Diariamente, 1 hora ou mais por dia": 4},
    "AM1": {"Há menos de 6 meses": 0, "6 a menos de 12 meses": 1, "Há 12 meses ou mais": 2, "Nunca": 3},
    "AM2": {"Há menos de 12 meses": 0, "Há 12 meses ou mais": 1, "Nunca": 2},
    "AM4": {"Há menos de 12 meses": 0, "Há 12 meses ou mais": 1, "Nunca": 2},
    "IN18": {"Nunca": 0, "Menos de uma vez por semana": 1, "Algumas vezes por semana": 2,
             "Todos os dias, apenas de manhã": 3, "Todos os dias, apenas antes de se deitar": 3,
             "Todos os dias, de manhã e antes de se deitar": 4,
             "Todos os dias, de manhã, após o almoço e antes de se deitar": 5},
    "IN19": {"Há menos de 10 anos": 0, "Entre 10 anos e menos de 20 anos": 1, "20 anos ou mais": 2, "Nunca": 3},
    "PA2": PA_3_5, "PA3": PA_3_5, "PA4": PA_3_5, "PA5": PA_2_3, "PA6": PA_5_10,
    "PA7_NAC": PA_2_3_5, "PA8_NAC": PA_2_3_5,
    "HATLEVEL": {"Nenhum nível completado": 0, "Básico 1º e 2º ciclo": 1, "Básico 3º ciclo": 2, "Secundário": 3,
                 "Pós secundário": 4, "C.T.Superior Profissional": 5, "Bacharelato ou licenciatura 1º ciclo": 6,
                 "Licenciatura pré-bolonha ou mestrado 2º ciclo": 7, "Doutoramento": 8},
    "HHINCOME": {"1º quintil": 1, "2º quintil": 2, "3º quintil": 3, "4º quintil": 4, "5º quintil": 5},
    "AW2_COD": {"1-7": 0, "8-14": 1, "15-30": 2, "31-180": 3, ">= 181": 4},
    "SS1": {"Nenhuma": 0, "1 ou 2": 1, "3 a 5": 2, "6 ou mais": 3},
    "SS2": {"Nenhuma preocupação e interesse": 0, "Pouca preocupação e interesse": 1,
            "Alguma preocupação e interesse": 2, "Muita preocupação e interesse": 3},
    "SS3": {"Muito difícil": 0, "Difícil": 1, "Possível": 2, "Fácil": 3, "Muito fácil": 4},
    "IC3": {"Menos de 10 horas": 0, "Entre 10 horas e menos de 20 horas": 1, "20 horas ou mais": 2},
    "IN13": {"Sozinho, sem dificuldade": 0, "Sozinho, mas com dificuldade": 1, "Só com ajuda": 2},
}

# Respostas que não cabem na escala mas são informação -> indicadora 0/1 (o valor fica NaN, ou 0 no caso do álcool)
ESPECIAIS = {
    "Não aplicável (nunca tentou ou não precisa de o fazer)": "nunca_tentou",
    "Não consumiu nos últimos 12 meses, por ter deixado de consumir álcool": "deixou_alcool",
    "Não consegue avaliar": "nao_consegue_avaliar",
}
SEM_VALOR = {"Não aplicável", "Sem resposta"}

In [8]:
def escolher_indicadoras(treino):
    # Decidido só com o treino: que colunas recebem indicadoras
    na = [c for c in ESCALAS if treino[c].eq("Não aplicável").mean() >= 0.01]
    especiais = [(c, texto, sufixo) for c in ESCALAS for texto, sufixo in ESPECIAIS.items()
                 if treino[c].eq(texto).any()]
    return na, especiais


def mapear_ordinais(df, na, especiais):
    # indicadoras ANTES de mapear (juntas de uma vez com pd.concat, mais rápido do que uma a uma)
    novas = {f"{c}_{sufixo}": df[c].eq(texto).astype("int8") for c, texto, sufixo in especiais}
    novas.update({f"{c}_nao_aplicavel": df[c].eq("Não aplicável").astype("int8") for c in na})
    df = pd.concat([df, pd.DataFrame(novas, index=df.index)], axis=1)
    for c, mapa in ESCALAS.items():
        s = df[c].astype("object")
        desconhecidos = set(s.dropna()) - set(mapa) - SEM_VALOR - set(ESPECIAIS)
        assert not desconhecidos, f"{c}: valores sem mapeamento {desconhecidos}"
        df[c] = s.map(mapa).astype("float64")                # "Não aplicável"/"Sem resposta" -> NaN
    return df

ORD_NAO_APLICAVEL, ORD_ESPECIAIS = escolher_indicadoras(train)      # decidido só com o treino
print("Colunas ordinais mapeadas:", len(ESCALAS))
print("Indicadoras 'Não aplicável':", len(ORD_NAO_APLICAVEL), "| indicadoras especiais:", len(ORD_ESPECIAIS))

Colunas ordinais mapeadas: 77
Indicadoras 'Não aplicável': 25 | indicadoras especiais: 10


## Passo 6 — Restante texto → `category`

As colunas que continuam a ser texto são categorias verdadeiras. Passam ao tipo `category`, que:
- os modelos de árvores do scikit-learn (ex: `HistGradientBoostingClassifier(categorical_features="from_dtype")`) tratam como categóricas;
- ocupa menos memória.

**As categorias são definidas no treino** e o teste usa exatamente as mesmas, pela mesma ordem.
Uma categoria que só exista no teste fica `NaN` (o modelo trata-a como valor em falta).

Este passo fica **fora** de `preparar` porque precisa de "aprender" as categorias com o treino.

In [9]:
def preparar(df):
    df = df.copy()                                           # não alterar o DataFrame original
    df = df.drop(columns=constantes + IDENTIFICADORES)       # passos 1 e 2
    df = df.replace("NR", "Sem resposta")                    # passo 3
    df = converter_numeros(df)                               # passo 4
    df = converter_pa1(df)
    df = mapear_ordinais(df, ORD_NAO_APLICAVEL, ORD_ESPECIAIS)   # passo 5
    return df


def para_categorias(treino, teste):
    treino, teste = treino.copy(), teste.copy()
    for c in treino.select_dtypes("object").columns:
        treino[c] = treino[c].astype("category")
        teste[c] = pd.Categorical(teste[c], categories=treino[c].cat.categories)
    return treino, teste


train_prep, test_prep = para_categorias(preparar(train), preparar(test))

print("Treino:", train.shape, "->", train_prep.shape)
print("Teste: ", test.shape, "->", test_prep.shape)
print()
print(train_prep.dtypes.map(lambda t: t.name).value_counts())

Treino: (10962, 237) -> (10962, 288)
Teste:  (3655, 236) -> (3655, 287)

category    121
float64     110
int8         54
int64         3
Name: count, dtype: int64


## Passo 7 — Verificar e gravar

In [10]:
# Verificações
assert not set(constantes + IDENTIFICADORES) & set(train_prep.columns), "ainda há colunas que deviam ter saído"
assert list(train_prep.columns.drop("AGE_RANGE")) == list(test_prep.columns), "treino e teste com colunas diferentes"
assert len(test_prep) == len(test), "o teste perdeu linhas"
assert train_prep.select_dtypes("object").empty, "ainda há colunas de texto"
for c in train_prep.select_dtypes("category").columns:
    assert list(train_prep[c].cat.categories) == list(test_prep[c].cat.categories), f"categorias diferentes em {c}"

# Categorias do teste que não existiam no treino (ficaram NaN)
novas = {c: int(test_prep[c].isna().sum() - test[c].isna().sum())
         for c in test_prep.select_dtypes("category").columns if c in test.columns}
print("Valores do teste com categoria desconhecida:", {c: n for c, n in novas.items() if n > 0})
print("OK")

Valores do teste com categoria desconhecida: {'IN62': 1, 'IN21': 1}
OK


In [11]:
# Resumo das colunas convertidas para número
convertidas = NUMERICAS + ["PA1_meses"]
resumo = train_prep[convertidas].describe().T[["count", "mean", "min", "50%", "max"]]
resumo["% NaN"] = (train_prep[convertidas].isna().mean() * 100).round(1)
resumo.round(1)

,count,mean,min,50%,max,% NaN
IN2,1433.0,12.8,1.0,10.0,61.0,86.9
IN3,1273.0,27.2,1.0,27.0,74.0,88.4
IN4,746.0,27.3,1.0,27.0,80.0,93.2
HHNBPERS_COD,10962.0,2.3,1.0,2.0,7.0,0.0
HHNBPERS_0_13_COD,10962.0,0.2,0.0,0.0,3.0,0.0
HO12_COD,10947.0,0.6,0.0,0.0,25.0,0.1
HO34_COD,10803.0,1.4,0.0,0.0,60.0,1.5
AM3,8168.0,0.6,0.0,0.0,12.0,25.5
AM5,5255.0,0.6,0.0,0.0,20.0,52.1
IN24,1476.0,2005.4,1969.0,2006.0,2019.0,86.5


In [12]:
# Gravar (formato parquet: guarda os tipos, incluindo as categorias)
train_prep.to_parquet("dados/train_tratado.parquet")
test_prep.to_parquet("dados/test_tratado.parquet")

# Confirmar que ao ler de volta fica tudo igual
volta = pd.read_parquet("dados/test_tratado.parquet")
assert volta.dtypes.equals(test_prep.dtypes)
for c in test_prep.select_dtypes("category").columns:
    assert list(volta[c].cat.categories) == list(test_prep[c].cat.categories)
print("Gravados: dados/train_tratado.parquet e dados/test_tratado.parquet")

Gravados: ../dados/train_tratado.parquet e ../dados/test_tratado.parquet
